# OnT quickstart: ontology embeddings in four steps

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HuiYang1997/OnT/blob/main/notebooks/OnT_Quickstart.ipynb)

**OnT** (Ontology Transformer, ISWC 2025) fine-tunes a sentence encoder so that concepts become points in a hyperbolic (Poincaré) ball:
general concepts sit near the centre, specific ones near the boundary, and $C \sqsubseteq D$ becomes "$C$ close to $D$, and further out".

1. **Install** the `ontology-transformer` package
2. **One click**: build a small pizza ontology and train OnT on it
3. **Embed** concepts, including $\exists r.C$
4. **Score** subsumptions $C \sqsubseteq D$ and rank candidate parents

Runs on the free Colab **CPU** runtime (a GPU is optional: *Runtime → Change runtime type*). Choose **Runtime → Run all**.

## Step 1 · Install

`ontology-transformer` reads OWL files through the OWL API, so it needs Java.

In [1]:
import os, sys, subprocess

if "google.colab" in sys.modules:                       # Java is not preinstalled on Colab
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

%pip install -q ontology-transformer==0.1.8

import logging, warnings           # keep the output short; delete these two lines to see all training logs
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)

Note: you may need to restart the kernel to use updated packages.


## Step 2 · One click: build an ontology and train OnT

The ontology below is the running example of the tutorial (axioms $\alpha_1$–$\alpha_6$), plus a few more pizzas and toppings.
We write it in OWL functional syntax; every class gets a readable label, because OnT encodes **labels**.

In [2]:
import re

def label(name):                       # "CheesyPizza" -> "cheesy pizza"
    return re.sub(r"(?<!^)(?=[A-Z])", " ", name).lower()

SUBCLASS = [                           # A ⊑ B
    ("Pizza", "Food"), ("DairyProduct", "Food"), ("Vegetable", "Food"), ("Meat", "Food"), ("Fruit", "Food"),
    ("Margherita", "Pizza"),                                  # α1
    ("Mozzarella", "Cheese"),                                 # α4
    ("Cheese", "DairyProduct"),                               # α5
    ("Parmesan", "Cheese"), ("Gorgonzola", "Cheese"), ("Ricotta", "Cheese"),
    ("Tomato", "Vegetable"), ("Mushroom", "Vegetable"), ("Onion", "Vegetable"), ("BellPepper", "Vegetable"),
    ("Ham", "Meat"), ("Salami", "Meat"), ("Chicken", "Meat"), ("Pineapple", "Fruit"),
    ("Funghi", "Pizza"), ("Hawaiian", "Pizza"), ("QuattroFormaggi", "Pizza"), ("Marinara", "Pizza"),
    ("Diavola", "Pizza"), ("Capricciosa", "Pizza"), ("Vegetariana", "Pizza"),
]
HAS_TOPPING = [                        # A ⊑ ∃hasTopping.B
    ("Margherita", "Mozzarella"), ("Margherita", "Tomato"),   # α2, α3
    ("Funghi", "Mozzarella"), ("Funghi", "Mushroom"),
    ("Hawaiian", "Ham"), ("Hawaiian", "Pineapple"),
    ("QuattroFormaggi", "Parmesan"), ("QuattroFormaggi", "Gorgonzola"), ("QuattroFormaggi", "Ricotta"),
    ("Marinara", "Tomato"),                                   # no cheese!
    ("Diavola", "Salami"), ("Diavola", "Mozzarella"),
    ("Capricciosa", "Ham"), ("Capricciosa", "Mushroom"), ("Capricciosa", "Mozzarella"),
    ("Vegetariana", "BellPepper"), ("Vegetariana", "Onion"), ("Vegetariana", "Mozzarella"),
]
DEFINED = [                            # A ≡ Pizza ⊓ ∃hasTopping.B
    ("CheesyPizza", "Cheese"),                                # α6
    ("MeatyPizza", "Meat"),
]

names = sorted({x for pair in SUBCLASS + HAS_TOPPING for x in pair} | {a for a, _ in DEFINED})
lines = ["Prefix(:=<http://example.org/pizza#>)",
         "Prefix(rdfs:=<http://www.w3.org/2000/01/rdf-schema#>)",
         "Ontology(<http://example.org/pizza>",
         "Declaration(ObjectProperty(:hasTopping))",
         'AnnotationAssertion(rdfs:label :hasTopping "has topping")']
for n in names:
    lines += [f"Declaration(Class(:{n}))", f'AnnotationAssertion(rdfs:label :{n} "{label(n)}")']
lines += [f"SubClassOf(:{a} :{b})" for a, b in SUBCLASS]
lines += [f"SubClassOf(:{a} ObjectSomeValuesFrom(:hasTopping :{b}))" for a, b in HAS_TOPPING]
lines += [f"EquivalentClasses(:{a} ObjectIntersectionOf(:Pizza ObjectSomeValuesFrom(:hasTopping :{b})))"
          for a, b in DEFINED]
lines.append(")")
open("pizza.ofn", "w").write("\n".join(lines))

print(f"pizza.ofn: {len(names)} classes, {len(SUBCLASS) + len(HAS_TOPPING) + len(DEFINED)} logical axioms")
print("\n".join(l for l in lines if l.startswith(("SubClassOf(:Margherita", "EquivalentClasses"))))

pizza.ofn: 29 classes, 46 logical axioms
SubClassOf(:Margherita :Pizza)
SubClassOf(:Margherita ObjectSomeValuesFrom(:hasTopping :Mozzarella))
SubClassOf(:Margherita ObjectSomeValuesFrom(:hasTopping :Tomato))
EquivalentClasses(:CheesyPizza ObjectIntersectionOf(:Pizza ObjectSomeValuesFrom(:hasTopping :Cheese)))
EquivalentClasses(:MeatyPizza ObjectIntersectionOf(:Pizza ObjectSomeValuesFrom(:hasTopping :Meat)))


Now the one line that matters: `fit` converts the OWL axioms into training data, fine-tunes the encoder
(default base model: `all-MiniLM-L12-v2`, 384 dimensions) and picks the best centripetal weight $\lambda$ for scoring.

In [3]:
from ont import OntologyTransformer

model = OntologyTransformer.fit(owl_path="pizza.ofn", output_dir="ont_pizza",
                                num_epochs=40, batch_size=8, learning_rate=5e-5)

print("best lambda:", model.best_lambda)

Download `en_core_web_sm` for pos tagger.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.8 MB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.7 MB/s  0:00:00


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Axiom Kind,Centri Weight,H@1,H@10,H@100,Mrr,Mr,Median,Auc
1,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.416667,2.500000,2.500000,0.955357
2,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.416667,2.500000,2.500000,0.955357
3,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.416667,2.500000,2.500000,0.955357
4,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.416667,2.500000,2.500000,0.955357
5,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.375000,3.000000,3.000000,0.946429
6,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.375000,3.000000,3.000000,0.946429
7,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143
8,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143
9,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143
10,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.982143


best lambda: 0.0


## Step 3 · Embed concepts

`encode` maps any label to a point in the Poincaré ball. Its hyperbolic distance from the origin acts as a **depth**:
general concepts should be closer to the centre than specific ones.
`encode_existence(r, C)` embeds the complex concept $\exists r.C$ by a learned rotation of $C$.

In [4]:
import pandas as pd
import torch

concepts = ["food", "pizza", "margherita", "dairy product", "cheese", "mozzarella"]
emb = model.encode(concepts, convert_to_tensor=True)
print("embedding matrix:", tuple(emb.shape))

depth = model.manifold.dist0(emb)              # hyperbolic distance to the origin
pd.DataFrame({"concept": concepts,
              "depth": depth.cpu().numpy().round(2),
              "first 4 coordinates": ["  ".join(f"{x:+.3f}" for x in v[:4]) for v in emb.cpu().numpy()]})

embedding matrix: (6, 384)


,concept,depth,first 4 coordinates
0,food,5.64,-0.119 +0.282 +0.047 +0.046
1,pizza,7.81,-0.429 +0.348 -0.081 +0.055
2,margherita,13.03,-0.303 +0.782 -0.178 +0.139
3,dairy product,7.94,-0.204 +0.198 +0.276 -0.043
4,cheese,8.63,-0.524 +0.212 -0.058 -0.062
5,mozzarella,9.50,-0.613 +0.279 -0.273 -0.009


In [5]:
some_cheese = model.encode_existence("has topping", "cheese")      # ∃hasTopping.Cheese
print("∃ has topping . cheese ->", some_cheese.shape, "  ".join(f"{x:+.3f}" for x in some_cheese[0, :4]), "...")

∃ has topping . cheese -> (1, 384) -0.572  +0.217  -0.070  -0.040 ...


## Step 4 · Score subsumptions

`score_hierarchy(C, D, λ)` returns $-\big(d(C,D) + \lambda\,(\lVert D\rVert - \lVert C\rVert)\big)$: **higher means $C \sqsubseteq D$ is more plausible**.
Compare candidate parents **of the same concept**: some are stated, some only *entailed* (never seen as such in training), some false.
On a toy ontology, stated parents win clearly, while distant entailed ancestors may score below a false parent: scores are plausibility, not proofs.

In [6]:
lam = model.best_lambda or 0.0

def score(child, parent):
    c = model.encode([child], convert_to_tensor=True)
    p = model.encode([parent], convert_to_tensor=True)
    return round(model.score_hierarchy(c, p, weight=lam).item(), 2)

rows = [("mozzarella", "cheese",        "stated (α4)"),
        ("mozzarella", "dairy product", "entailed (α4, α5)"),
        ("mozzarella", "pizza",         "false"),
        ("margherita", "pizza",         "stated (α1)"),
        ("margherita", "food",          "entailed"),
        ("margherita", "cheese",        "false")]
pd.DataFrame([(f"{c} ⊑ {p}", status, score(c, p)) for c, p, status in rows],
             columns=["axiom", "status", "score"])

,axiom,status,score
0,mozzarella ⊑ cheese,stated (α4),-5.51
1,mozzarella ⊑ dairy product,"entailed (α4, α5)",-9.91
2,mozzarella ⊑ pizza,false,-6.37
3,margherita ⊑ pizza,stated (α1),-9.55
4,margherita ⊑ food,entailed,-11.69
5,margherita ⊑ cheese,false,-11.39


The same score ranks **all** classes as candidate parents, for an existing class or for a concept that is not in the ontology yet:

In [7]:
all_labels = [label(n) for n in names]
cand = model.encode(all_labels, convert_to_tensor=True)

def top_parents(query, k=4):
    q = model.encode([query], convert_to_tensor=True)
    s = model.score_hierarchy(q.expand_as(cand), cand, weight=lam)
    return [all_labels[i] for i in s.argsort(descending=True).tolist() if all_labels[i] != query][:k]

for query in ["mozzarella", "margherita", "pepperoni pizza"]:      # "pepperoni pizza" is not in the ontology
    print(f"{query:>16} ⊑ ?  ", top_parents(query))

      mozzarella ⊑ ?   ['cheese', 'pizza', 'vegetariana', 'marinara']
      margherita ⊑ ?   ['pizza', 'mozzarella', 'vegetariana', 'marinara']
 pepperoni pizza ⊑ ?   ['pizza', 'bell pepper', 'meaty pizza', 'cheese']


### Notes

* Scores are **plausibility**, not proofs: check any new axiom with an OWL reasoner (e.g. ELK, HermiT).
* **Your own ontology**: replace `pizza.ofn` by any OWL file (`.owl`, `.ofn`, `.ttl`), e.g. uploaded via the Colab file panel.
* **No training**: load a released model, e.g. `OntologyTransformer.from_pretrained("Hui97/OnT-MiniLM-L12-galen")` (also `-go`, `-anatomy`, and `OnT-MPNet-*`).
* Paper: Yang, Chen, He, Gao & Horrocks. *Language Models as Ontology Encoders*. ISWC 2025. Code: github.com/HuiYang1997/OnT